<a href="https://colab.research.google.com/github/BardRimon/Study/blob/main/CV/HW3_ImageClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее  задание
Ознакомиться с одной из научных статей (по вариантам):

* [Physiological Inspired Deep Neural Networks for Emotion Recognition](https://ieeexplore.ieee.org/stamp/stamp.jsp?arnumber=8472816&tag=1)

* [Boundary loss for highly unbalanced segmentation](https://arxiv.org/abs/1812.07032)


* [Correlation Maximized Structural Similarity Loss for Semantic Segmentation](https://arxiv.org/abs/1910.08711)

* [Topology-Preserving Deep Image Segmentation](https://papers.nips.cc/paper/8803-topology-preserving-deep-image-segmentation)



На основе выбранной статьи:

1. реализовать функцию потерь которая описана в статье,
2. описать её математическую формулу и интуицию работы,
3. объяснить, какие проблемы она решает (например, дисбаланс классов, сохранение границ, сохранение топологии и т. д.).
4. реализовать одну модель сегментации:
(**варианты: LinkNet, U-Net, DeepLab v1, PSPNet**);
5. провести обучение и сравнение результатов с базовыми функциями потерь
(BCE, Dice, Focal, Tversky) на одном и том же датасете.
6. Создать гибридную функцию потерь —
объединить предложенный лосс с одной из классических (например, Boundary + Dice, SSIM + Focal) и сравнить динамику сходимости и итоговые метрики (IoU, Dice, Precision, Recall).


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PhysiologicalEmotionLoss(nn.Module):
    """
    Implementation of the loss function described in 'Physiological Inspired Deep Neural Networks
    for Emotion Recognition'.

    This loss includes:
    1. [cite_start]Classification Loss (Categorical Cross-Entropy) [cite: 217]
    2. Facial Parts Loss, which can be:
       - [cite_start]Fully Supervised (MSE with target maps) [cite: 228]
       - [cite_start]Weakly Supervised (Sparsity + Spatial Contiguity) [cite: 247]
       - [cite_start]Hybrid (Combination of both) [cite: 272]
    """

    def __init__(self,
                 lambda_main=1.0,
                 lambda_sparsity=1e-4,
                 gamma_contiguity=1.0,
                 mode='hybrid'):
        """
        Args:
            lambda_main (float): Weight λ controlling interaction between classification
                                 [cite_start]and facial parts loss (Eq. 2)[cite: 216].
            [cite_start]lambda_sparsity (float): Weight λ for the sparsity term in weakly supervised mode (Eq. 5)[cite: 364].
            [cite_start]gamma_contiguity (float): Weight γ for the contiguity term in weakly supervised mode (Eq. 5)[cite: 257].
            mode (str): One of 'supervised', 'weakly', or 'hybrid'.
        """
        super(PhysiologicalEmotionLoss, self).__init__()
        self.lambda_main = lambda_main
        self.lambda_sparsity = lambda_sparsity
        self.gamma_contiguity = gamma_contiguity
        self.mode = mode

        # [cite_start]Classification loss is Categorical Cross-Entropy (Eq. 3) [cite: 217]
        self.classification_criterion = nn.CrossEntropyLoss()

        # [cite_start]Fully supervised loss is Mean Squared Error (Eq. 4) [cite: 243]
        self.mse_criterion = nn.MSELoss()

    def _sparsity_loss(self, pred_map):
        """
        [cite_start]Calculates the sparsity term (L1 regularization) defined in Eq. 6[cite: 260].
        L_sparsity = (1 / m*n) * sum(|x_ij|)
        """
        # Calculate mean of absolute values (equivalent to sum divided by m*n pixels)
        return torch.mean(torch.abs(pred_map))

    def _contiguity_loss(self, pred_map):
        """
        Calculates the spatial contiguity term (Total Variation) defined in Eq. [cite_start]7[cite: 266].
        Minimizes local spatial transitions to encourage smoothness.
        """
        # pred_map shape: (batch, channels, height, width)
        h, w = pred_map.shape[2], pred_map.shape[3]

        # Calculate differences between adjacent pixels in height and width dimensions
        diff_h = torch.abs(pred_map[:, :, 1:, :] - pred_map[:, :, :-1, :])
        diff_w = torch.abs(pred_map[:, :, :, 1:] - pred_map[:, :, :, :-1])

        # Sum of differences normalized by image resolution (m * n)
        # Note: We sum over the spatial dimensions and divide by total pixels
        loss_h = torch.sum(diff_h) / (h * w * pred_map.shape[0] * pred_map.shape[1])
        loss_w = torch.sum(diff_w) / (h * w * pred_map.shape[0] * pred_map.shape[1])

        return loss_h + loss_w

    def forward(self, pred_class, target_class, pred_map, target_map=None):
        """
        Args:
            pred_class (Tensor): Predicted logits for expression classes (y_hat).
            target_class (Tensor): Ground truth class indices.
            [cite_start]pred_map (Tensor): Predicted relevance map (x_hat) from Facial Parts Component[cite: 165].
            target_map (Tensor, optional): Target relevance map (x_target) generated from landmarks.
                                           [cite_start]Required for 'supervised' and 'hybrid' modes[cite: 239].
        """
        # [cite_start]1. Classification Loss (Eq. 3) [cite: 219]
        loss_classification = self.classification_criterion(pred_class, target_class)

        loss_facial_parts = 0.0

        # 2. Facial Parts Loss
        # [cite_start]Fully Supervised Component (Eq. 4) [cite: 243]
        if self.mode in ['supervised', 'hybrid']:
            if target_map is None:
                raise ValueError("target_map cannot be None for supervised or hybrid mode.")
            loss_supervised = self.mse_criterion(pred_map, target_map)
            loss_facial_parts += loss_supervised

        # [cite_start]Weakly Supervised Component (Eq. 5, 6, 7) [cite: 249, 260, 266]
        if self.mode in ['weakly', 'hybrid']:
            l_sparsity = self._sparsity_loss(pred_map)
            l_contiguity = self._contiguity_loss(pred_map)

            # [cite_start]Weighted sum for weakly supervised part (Eq. 5) [cite: 256]
            loss_weakly = (self.lambda_sparsity * l_sparsity) + (self.gamma_contiguity * l_contiguity)

            # If hybrid, we combine them. [cite_start]The paper suggests weighted summation (Section III-B-3)[cite: 274].
            # Assuming simple addition logic based on "weighted summation of loss terms defined in Eq 4 and 5"
            loss_facial_parts += loss_weakly

        # [cite_start]3. Total Loss (Eq. 2) [cite: 214]
        # L = L_classification + lambda * L_facial_parts
        total_loss = loss_classification + (self.lambda_main * loss_facial_parts)

        return total_loss

### Общая формулировка
Функция потерь $L$ является составной. Она обучает модель одновременно классифицировать эмоции и определять «карту релевантности» (relevance map), которая подсвечивает важные участки лица.

Формула выглядит следующим образом:
$$L = \mathcal{L}_{classification} + \lambda \mathcal{L}_{facial\_parts}$$
где $\lambda \ge 0$ — гиперпараметр, контролирующий баланс между задачей классификации и задачей выделения признаков лица.

---

### 1. Компонент классификации ($\mathcal{L}_{classification}$)
**Формула:** Используется стандартная категориальная кросс-энтропия (Categorical Cross-Entropy):
$$\mathcal{L}_{classification} = -\sum_{i=1}^{N} y_i \log(\hat{y}_i)$$
где $y_i$ — истинная метка класса (one-hot вектор), а $\hat{y}_i$ — предсказанная вероятность, полученная после softmax.

**Интуиция:** Этот член заставляет сеть предсказывать правильную эмоцию (например, радость или злость) на основе признаков, извлеченных из изображения.

---

### 2. Компонент лицевых частей ($\mathcal{L}_{facial\_parts}$)
Этот компонент отвечает за обучение «карты релевантности» $\hat{x}$. Его цель — заставить сеть фокусироваться только на тех участках лица, которые физиологически участвуют в выражении эмоций (мышцы, глаза, рот).

В зависимости от режима (Fully Supervised, Weakly Supervised или Hybrid), этот компонент рассчитывается по-разному.

#### А. Полностью контролируемый режим (Fully Supervised)
**Формула:** Используется среднеквадратичная ошибка (MSE) между предсказанной картой релевантности $\hat{x}$ и целевой картой $x^{target}$:
$$\mathcal{L}_{facial\_parts} = \frac{1}{N}\sum_{i=1}^{N}(x_{i}^{target}-\hat{x}_{i})^{2}$$
**Интуиция:** Если у нас есть разметка ключевых точек лица (landmarks), мы создаем идеальную «тепловую карту» ($x^{target}$), состоящую из гауссиан вокруг глаз, носа и рта. Мы прямо учим сеть активироваться именно в этих зонах.

#### Б. Слабо контролируемый режим (Weakly Supervised)
Этот режим используется, когда нет разметки ключевых точек. Он опирается на физиологическую гипотезу о том, что важные регионы лица — это небольшие (разреженные) и цельные (непрерывные) области.

**Формула:**
$$\mathcal{L}_{facial\_parts} = \lambda_{sparsity} \mathcal{L}_{sparsity}(\hat{x}) + \gamma \mathcal{L}_{contiguity}(\hat{x})$$

1.  **Разреженность (Sparsity, $\mathcal{L}_{sparsity}$):**
    * *Математика:* L1-регуляризация активаций карты.
  * $\mathcal{L}_{sparsity} \hat{x}=\frac{1}{m \times n}\sum_{i,j}|\hat{x}_{i,j}|$
    * *Интуиция:* Большинство пикселей на лице не важны для эмоции. Важны только конкретные мышцы. L1-норма заставляет карту быть «пустой» (нули) почти везде, кроме самых важных мест.

2.  **Пространственная непрерывность (Spatial Contiguity, $\mathcal{L}_{contiguity}$):**
    * *Математика:* Total Variation (минимизация разницы между соседними пикселями).
  * $\mathcal{L}_{contiguity}(\hat{x})=\frac{1}{m \times n}\sum |\hat{x}_{i+1,j}-\hat{x}_{i,j}|+|\hat{x}_{i,j+1}-\hat{x}_{i,j}|$


    * *Интуиция:* Активации не должны быть шумными (разбросанными пикселями). Мышцы — это сплошные объекты. Этот член сглаживает карту, заставляя активные регионы быть локализованными пятнами, а не шумом.

#### В. Гибридный режим (Hybrid)
**Формула:** Взвешенная сумма полностью контролируемого и слабо контролируемого методов.
**Интуиция:** Комбинирует сильные стороны обоих подходов. MSE гарантирует, что сеть найдет основные черты (глаза, рот), а слабый контроль (разреженность + непрерывность) позволяет сети самой найти дополнительные важные детали, такие как морщины от эмоций или ямочки, которые не отмечены в стандартных landmarks.

### Итоговая суть (Summary)
Aвторы предлагают архитектуру, которая не просто классифицирует картинку целиком, а сначала учится понимать, *куда* смотреть (с помощью $\mathcal{L}_{facial\_parts}$), и использует эту информацию для усиления признаков в нужных местах перед классификацией.